# 07. Huấn luyện GCN, GraphSAGE, GAT nhẹ (task 5.2, 5.3)

Đọc đồ thị từ output của kernel `gnn-iot-ids-04-build-graphs`. Mỗi mô hình chạy 3 seed 42, 43, 44, chọn epoch theo F1-macro (4 lớp chính) trên **val**; chỉ ghi `split=val`. Độ trễ đo trên CPU 1 luồng.

In [ ]:
import glob, os, subprocess, sys

def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-9000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode != 0:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")

REPO = "/tmp/gnn-iot-ids"
run(["rm", "-rf", REPO])
run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
run(["git", "-C", REPO, "log", "-1", "--format=%H %s"])
with open(f"{REPO}/requirements.txt") as f:
    reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
run([sys.executable, "-m", "pip", "install", "-q", *keep])
run(["nvidia-smi", "-L"])

In [ ]:
OUT = "/kaggle/working"
for ds in ("nf_ton_iot_v3", "nf_ton_iot_v2"):
    d = os.path.dirname(next(glob.iglob(f"/kaggle/input/**/{ds}/split.json", recursive=True)))
    print("=====", ds, d)
    run([sys.executable, "-I", "scripts/run_gnn.py", d, ds, f"{OUT}/results_gnn_{ds}.csv", f"{OUT}/ckpt"],
        cwd=REPO)